In [ ]:
import os
os.environ['PYTHONUTF8'] = '1'

print("✅ Environment ready")

In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
%pip install transformers accelerate peft trl bitsandbytes numpy

print("✅ Environment ready")

In [ ]:
import json

with open('cefr_training_dataset.json', encoding='utf-8') as f:
    data = json.load(f)

print(f"✅ Loaded {len(data)} examples")

In [ ]:
from datasets import Dataset

formatted_data = []

for example in data:
    text = f"""[SYSTEM]
{example['system']}

[USER]
{example['user']}

[ASSISTANT]
{example['assistant']}"""

    formatted_data.append(text)

hf_dataset = Dataset.from_dict({'text': formatted_data})

train_test = hf_dataset.train_test_split(test_size=0.2, seed=42)

print(f"✅ Train: {len(train_test['train'])}")
print(f"✅ Test: {len(train_test['test'])}")

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch
from huggingface_hub import login

login(token="")

model_name = "meta-llama/Llama-2-7b-hf"

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

print("✅ Model loaded")

In [ ]:
from transformers import AutoTokenizer

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.model_max_length = 128

# tokenize dataset
train_dataset = train_test["train"].map(
    lambda x: tokenizer(
        x["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    ),
    batched=True
)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./lora_adapter",
    num_train_epochs=2,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=5e-5,
    max_grad_norm=1.0,

    logging_steps=10,
    save_steps=100,

    eval_strategy="no",
    save_strategy="steps",

    fp16=False,
    bf16=True
)


In [ ]:
# ✅ Train model
from trl import SFTTrainer

print("Starting training...")

trainer = SFTTrainer(
    model=model,
    train_dataset=train_test['train'],
    args=training_args,
    processing_class=tokenizer
)

torch.backends.cuda.matmul.allow_tf32 = True

trainer.train()

print("✅ Training complete")

In [ ]:
print("Merging LoRA...")
merged_model = model.merge_and_unload()


# ✅ Save merged model
merged_model.save_pretrained("./merged_model")
tokenizer.save_pretrained("./merged_model")

print("✅ Merged model saved")